# 🌉 Minimum Spanning Tree — Runnable Notebook

Companion to [`README.md`](README.md) and
[`09_minimum_spanning_tree_lesson.html`](09_minimum_spanning_tree_lesson.html).

Connect all vertices with the **least total edge weight**, no cycles — via **Kruskal**, **Prim**, and **Union-Find**.

## 1. Union-Find (Disjoint Set Union)
The engine that answers *'would this edge form a cycle?'* in near-O(1).

In [ ]:
class UnionFind:
    """Tracks which group each element is in, with two speed tricks."""
    def __init__(self, n):
        self.parent = list(range(n))       # each element is its own leader at first
        self.rank = [0] * n                # tree-height hint

    def find(self, x):
        """Group leader of x, flattening the path as we climb (path compression)."""
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        """Merge groups of a and b. Returns False if they were ALREADY together."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                   # same group -> this edge would make a cycle
        if self.rank[ra] < self.rank[rb]:  # union by rank: hang shorter under taller
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

uf = UnionFind(5)
print("union(0,2):", uf.union(0, 2))   # True (merged)
print("union(1,2):", uf.union(1, 2))   # True (merged into {0,1,2})
print("union(0,1):", uf.union(0, 1))   # False (already together -> a cycle)
assert uf.find(0) == uf.find(1) == uf.find(2)

## 2. Kruskal — cheapest edges that don't form a cycle

In [ ]:
def kruskal(n, edges):
    """edges = list of (weight, u, v). Returns (mst_edges, total_weight)."""
    uf = UnionFind(n)
    mst, total = [], 0
    for w, u, v in sorted(edges):          # cheapest first
        if uf.union(u, v):                 # different groups -> safe, no cycle
            mst.append((u, v, w))
            total += w
            if len(mst) == n - 1:          # a spanning tree has exactly V-1 edges
                break
    return mst, total

#   A=0 B=1 C=2 D=3 E=4  (undirected, weighted)
edges = [(1, 0, 2), (2, 1, 2), (3, 3, 4), (4, 0, 1), (5, 1, 3), (6, 2, 4)]
mst_k, total_k = kruskal(5, edges)
print("Kruskal MST edges:", mst_k)
print("Kruskal total    :", total_k)
assert total_k == 11 and len(mst_k) == 4

## 3. Prim — grow one tree, always take the cheapest frontier edge

In [ ]:
import heapq

def prim(adj, start=0):
    """adj[u] = list of (v, weight). Returns (mst_edges, total_weight)."""
    visited = {start}
    pq = [(w, start, v) for v, w in adj[start]]   # edges leaving the start vertex
    heapq.heapify(pq)
    mst, total = [], 0
    while pq and len(visited) < len(adj):
        w, u, v = heapq.heappop(pq)        # cheapest edge crossing the frontier
        if v in visited:
            continue                       # v already in the tree -> skip
        visited.add(v)
        mst.append((u, v, w))
        total += w
        for nxt, w2 in adj[v]:             # v's edges become new frontier options
            if nxt not in visited:
                heapq.heappush(pq, (w2, v, nxt))
    return mst, total

# same graph as an adjacency list
adj = {0: [(2, 1), (1, 4)], 1: [(2, 2), (0, 4), (3, 5)],
       2: [(0, 1), (1, 2), (4, 6)], 3: [(4, 3), (1, 5)], 4: [(3, 3), (2, 6)]}
mst_p, total_p = prim(adj, 0)
print("Prim MST edges:", mst_p)
print("Prim total    :", total_p)
assert total_p == 11                       # same minimum as Kruskal
print("Kruskal and Prim agree on the total:", total_k == total_p)

## 4. Cycle check with Union-Find (the reusable trick)

In [ ]:
def has_cycle_uf(n, undirected_edges):
    """Process each edge; if endpoints already share a group, it's a cycle."""
    uf = UnionFind(n)
    for u, v in undirected_edges:
        if not uf.union(u, v):             # union returns False -> already connected
            return True
    return False

print("triangle 0-1-2-0 has a cycle?", has_cycle_uf(3, [(0, 1), (1, 2), (2, 0)]))
print("path 0-1-2 has a cycle?      ", has_cycle_uf(3, [(0, 1), (1, 2)]))
assert has_cycle_uf(3, [(0, 1), (1, 2), (2, 0)])
assert not has_cycle_uf(3, [(0, 1), (1, 2)])

## ✅ Recap
- **MST** = connect all `V` vertices, no cycle, `V−1` edges, **minimum total weight**.
- **Union-Find**: `find` (path compression) + `union` (by rank) ≈ `O(α(n))`; `union`→False means a cycle.
- **Kruskal**: sort edges, add cheapest non-cycling → `O(E log E)`, great for **sparse**.
- **Prim**: grow a tree via a min-heap of frontier edges → `O((V+E) log V)`, great for **dense**.

Next: [`10_Topological_Sort`](../10_Topological_Sort/README.md).